# 04_data_profile_CHS

- 목적: 모델링 전 데이터 프로파일 — B-1 타임스탬프 무결성, B-2 정상 의심 세그먼트(31 오류분석 FP 후보), E3 정상 하루 안 추이·드리프트, E4 이상 이벤트 추이 재검(AC 기준, 02 §4 정정). 윈도우 단위 p값은 보고하지 않고 세그먼트 대표값으로만 검정한다.
- 입력: `data/processed/11_window_features.parquet`(`win_s==1.0`), `data/processed/segments.csv`, 원시 `data_quality.load()`(B-1 타임스탬프 검사에만)
- 출력: `figures/04_data_profile_CHS/`, `results/04_data_profile_CHS/` (`timestamp_integrity.csv`, `normal_suspects.csv`, `normal_trend.csv`, `block_drift_ks.csv`, `block_state_share.csv`, `state_transitions.csv`, `outlier_trend_ac.csv`)
- 담당: CHS
- 심사 대응: 1번(타임스탬프·라벨 신뢰도 진단), 3번(FP 후보·FN 이원화 근거), 4번(재적합 주기·부하 이동 근거)

> 이 노트북은 **진단 목적**으로 `src`·`label`·`t_abs`·`state`·`*_dc` 열을 쓴다. 절대 시간(`t_abs`·`start`)은 **정렬과 추세 x축으로만** 쓰며 모델 입력 피처가 아니다. 근거 수준은 [데이터]/[추정]/[가정]로 표기한다.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
NB = "04_data_profile_CHS"  # 파일명(확장자 제외)과 같게
FIG, RES = paths.nb_dirs(NB)
SEED = 42
np.random.seed(SEED)
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False
import data_quality, split, stats
PQ = paths.DATA_PROCESSED / "11_window_features.parquet"
SEG = paths.DATA_PROCESSED / "segments.csv"
win = pd.read_parquet(PQ); win = win[win["win_s"] == 1.0].reset_index(drop=True)
seg = pd.read_csv(SEG); seg["start"] = pd.to_datetime(seg["start"])
print(NB, FIG.relative_to(ROOT), RES.relative_to(ROOT))
print("windows(1s):", len(win), win["label"].value_counts().to_dict(), "| segments:", seg["label"].value_counts().to_dict())
print(seg.loc[seg.label == 1, ["seg_uid", "seg_id", "len", "state"]].head(3).to_dict("records"))

04_data_profile_CHS figures\04_data_profile_CHS results\04_data_profile_CHS
windows(1s): 3360 {0: 3264, 1: 96} | segments: {0.0: 599, 1.0: 21}
[{'seg_uid': 'outlier_0', 'seg_id': 0, 'len': 4, 'state': nan}, {'seg_uid': 'outlier_1', 'seg_id': 1, 'len': 50, 'state': nan}, {'seg_uid': 'outlier_2', 'seg_id': 2, 'len': 47, 'state': nan}]


## 1. B-1 타임스탬프 무결성

**가설**: `TimeStamp`는 단조 증가하고 burst 안 샘플 간격이 0.1 s(10 Hz)라, 03 A-1의 에일리어싱 계산(60 Hz → 0.6 Hz)이 전제하는 fs = 10.00 Hz가 성립한다. 01 리포트의 "중복 1건"은 같은 세그먼트 안 같은 값의 단순 중복이다.

**방법**: 정상·이상 원시 파일 각각에서 단조성, 역행(dt<0)·동일 시각(dt=0) 건수, 중복 시각 행의 세그먼트·값 동일 여부, burst 내부 dt(세그먼트 시작 행 제외) 분포(0.1 s ±5% 밖 비율, 0.15 s 초과 건수), 세그먼트별 평균 dt → 실효 fs 중앙값. 판정 기준: 실효 fs가 10.00 Hz에서 0.5% 이상 벗어나면 에일리어싱 후보 주파수 재계산이 필요하다.

In [2]:
SENS = data_quality.SENSORS
def ts_profile(key):
    d = data_quality.load(key); dt = d["dt"]
    inb = dt[(d["seg"] == d["seg"].shift()) & dt.notna()]          # burst 내부 간격(세그먼트 첫 행 제외)
    dup = d[d["ts"].duplicated(keep=False)]
    dup_same = bool(len(dup) and dup.groupby("ts")["seg"].nunique().max() == 1 and dup.groupby("ts")[SENS].nunique().max().max() == 1)
    g = d.groupby("seg")["ts"]; n = g.size(); span = (g.max() - g.min()).dt.total_seconds()
    fs = (1.0 / (span / (n - 1))[n >= 2]).replace([np.inf], np.nan).dropna()
    return {"file": key, "n_rows": len(d), "n_segments": int(d["seg"].nunique()), "monotonic_increasing": bool(d["ts"].is_monotonic_increasing),
            "n_backward": int((dt < 0).sum()), "n_zero_dt": int((dt == 0).sum()), "n_dup_ts_rows": int(d["ts"].duplicated().sum()),
            "dup_same_segment_and_values": dup_same if len(dup) else None,
            "n_inburst_dt": len(inb), "dt_median_s": float(inb.median()), "dt_mean_s": float(inb.mean()),
            "frac_outside_0.1pm5pct": float(((inb < 0.095) | (inb > 0.105)).mean()), "n_dt_gt_0.15s": int((inb > 0.15).sum()),
            "fs_eff_median_hz": float(fs.median()), "fs_dev_pct": float((fs.median() / 10.0 - 1) * 100), "n_seg_for_fs": len(fs)}, inb
res = [ts_profile(k) for k in data_quality.FILES]
ts_tbl = pd.DataFrame([r[0] for r in res]); ts_tbl["needs_alias_recalc(|dev|>=0.5%)"] = ts_tbl["fs_dev_pct"].abs() >= 0.5
ts_tbl.to_csv(RES / "timestamp_integrity.csv", index=False)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (r, inb), k in zip(axs, res, data_quality.FILES):
    ax.hist(inb.clip(upper=0.4), bins=80); ax.set_yscale("log"); ax.axvspan(0.095, 0.105, color="g", alpha=.2)
    ax.set_title(f"{k}: burst 내부 dt (>0.4 s는 0.4로 절단)"); ax.set_xlabel("dt (s)")
paths.save_fig(FIG, "sample_interval_hist.png")
print(ts_tbl.T.to_string())

saved figures\04_data_profile_CHS\sample_interval_hist.png
                                        0        1
file                               normal  outlier
n_rows                              20000      600
n_segments                            599       21
monotonic_increasing                 True     True
n_backward                              0        0
n_zero_dt                               1        0
n_dup_ts_rows                           1        0
dup_same_segment_and_values          True     None
n_inburst_dt                        19401      579
dt_median_s                           0.1      0.1
dt_mean_s                        0.099995      0.1
frac_outside_0.1pm5pct           0.000052      0.0
n_dt_gt_0.15s                           0        0
fs_eff_median_hz                     10.0     10.0
fs_dev_pct                            0.0      0.0
n_seg_for_fs                          590       21
needs_alias_recalc(|dev|>=0.5%)     False    False


**판정: 채택** (`timestamp_integrity.csv`)

| 항목 | 정상 | 이상 |
|---|---|---|
| 행 수 / 세그먼트 | 20,000 / 599 | 600 / 21 |
| 단조 증가 · 역행(dt<0) | 예 · 0건 | 예 · 0건 |
| 동일 시각(dt=0) | 1건 (같은 세그먼트·센서 값 동일) | 0건 |
| burst 내부 dt 중앙값 | 0.1 s | 0.1 s |
| 0.1 s ±5% 밖 비율 | 0.0052% (19,401개 중 1개) | 0% (579개) |
| dt > 0.15 s | 0건 | 0건 |
| 실효 fs 중앙값 (세그먼트별 평균 dt) | 10.0000 Hz (편차 0.0%) | 10.0000 Hz (편차 0.0%) |

- 01 리포트의 "중복 1건"은 동일 세그먼트 안 값까지 같은 중복 행 1개다 [데이터]. 세그먼트 경계 오분할·값 충돌이 아니다.
- 실효 fs 편차가 0.5% 기준에 한참 못 미치므로(두 파일 모두 0.0%) 03 A-1의 에일리어싱 계산(60 Hz → 0.6 Hz)은 fs = 10.00 Hz 전제로 재계산할 필요가 없다 [데이터].
- **모델링 영향**: 중복 1행은 윈도우 피처에 영향이 없다(1초 윈도우 10샘플 중 1행). 시간 정렬·윈도우 구성에 별도 전처리 불필요. 심사 1번 "데이터 신뢰도" 서술에 근거로 쓴다.

## 2. B-2 정상 의심 세그먼트

**가설**: 정상 599 세그먼트 중 일부는 이상 확실군 수준의 진동을 보이는 "정상 이상치"이며, 31 오류분석에서 FP로 나타날 것이다.

**방법**: (a) `segments.csv`의 `vib_rms`가 02 §1 이상 "확실 이상" 최솟값(0.258)을 넘는 것, (b) 세그먼트 `AI0_Vibration_rms`가 정상 세그먼트 q99를 넘는 것, (c) 윈도우 parquet 1초 AI0 rms의 세그먼트 최댓값이 정상 윈도우 q99를 넘는 것. 세 기준의 합집합을 `normal_suspects.csv`로 저장한다.

In [3]:
SURE_MIN = 0.258
nseg = seg[seg.label == 0].copy()
q99_seg = nseg["AI0_Vibration_rms"].quantile(0.99)
nwin = win[win.label == 0]; q99_win = nwin["AI0_Vibration_rms"].quantile(0.99)
wmax = nwin.groupby("seg_uid")["AI0_Vibration_rms"].max().rename("win_ai0_rms_max")
nseg = nseg.merge(wmax, on="seg_uid", how="left")
nseg["c_vib_rms"] = nseg["vib_rms"] > SURE_MIN
nseg["c_seg_ai0_q99"] = nseg["AI0_Vibration_rms"] > q99_seg
nseg["c_win_ai0_q99"] = nseg["win_ai0_rms_max"] > q99_win
crit = ["c_vib_rms", "c_seg_ai0_q99", "c_win_ai0_q99"]
nseg["criteria"] = nseg[crit].apply(lambda r: "+".join(c[2:] for c in crit if r[c]), axis=1)
blk = split.block_assignment(nseg, 5).rename("block"); nseg = nseg.merge(blk, left_on="seg_uid", right_index=True, how="left")
sus = nseg[nseg[crit].any(axis=1)].sort_values("start")
cols = ["seg_uid", "seg_id", "start", "len", "state", "vib_rms", "AI0_Vibration_rms", "AI2_Current_rms", "criteria", "block"]
sus[cols].to_csv(RES / "normal_suspects.csv", index=False)
print(f"vib_rms 기준 {SURE_MIN} / 세그먼트 AI0 q99 {q99_seg:.4f} / 윈도우 AI0 q99 {q99_win:.4f}")
print("기준별:", {c: int(nseg[c].sum()) for c in crit}, "| 합집합", len(sus), "| 3기준 모두", int(nseg[crit].all(axis=1).sum()))
print("의심 상태 분포:", sus["state"].value_counts().to_dict(), "| 정상 전체 상태 비율:", nseg["state"].value_counts(normalize=True).round(3).to_dict())
print("의심 블록 분포:", sus["block"].value_counts().sort_index().to_dict(), "| 정상 블록별 세그먼트 수:", nseg["block"].value_counts().sort_index().to_dict())
print(sus[cols].head(15).to_string(index=False))
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.scatter(nseg["start"], nseg["AI0_Vibration_rms"], s=8, c=np.where(nseg["state"] == 1, "tab:blue", "tab:orange"), alpha=.5)
ax.scatter(sus["start"], sus["AI0_Vibration_rms"], s=45, facecolors="none", edgecolors="r", label=f"의심 {len(sus)}개")
ax.axhline(q99_seg, ls="--", c="gray", label="정상 세그먼트 AI0 rms q99")
ax.set_xlabel("세그먼트 시작(정렬용 x축)"); ax.set_ylabel("AI0 rms"); ax.legend(); ax.set_title("정상 세그먼트 AI0 rms와 의심 세그먼트 (파랑 state1 / 주황 state0)")
paths.save_fig(FIG, "normal_segment_outliers.png")

vib_rms 기준 0.258 / 세그먼트 AI0 q99 0.1177 / 윈도우 AI0 q99 0.1272
기준별: {'c_vib_rms': 1, 'c_seg_ai0_q99': 6, 'c_win_ai0_q99': 21} | 합집합 26 | 3기준 모두 0
의심 상태 분포: {0.0: 25, 1.0: 1} | 정상 전체 상태 비율: {1.0: 0.579, 0.0: 0.421}
의심 블록 분포: {0: 6, 1: 1, 2: 8, 3: 10, 4: 1} | 정상 블록별 세그먼트 수: {0: 120, 1: 120, 2: 120, 3: 120, 4: 119}
   seg_uid  seg_id                   start  len  state  vib_rms  AI0_Vibration_rms  AI2_Current_rms                criteria  block
 normal_33      33 2022-07-12 00:04:25.827   50    0.0 0.166478           0.081810       139.181245             win_ai0_q99      0
 normal_49      49 2022-07-12 00:06:16.934   40    0.0 0.138794           0.087637       127.896851             win_ai0_q99      0
 normal_52      52 2022-07-12 00:06:36.231   50    0.0 0.163983           0.093953       122.422251             win_ai0_q99      0
 normal_67      67 2022-07-12 00:08:26.424   50    0.0 0.151839           0.091344       126.071097             win_ai0_q99      0
 normal_94      94 2022-07-12 00:1

saved figures\04_data_profile_CHS\normal_segment_outliers.png


**판정: 채택** (`normal_suspects.csv`, 26행) — **31 오류분석의 FP 후보 목록**으로 쓴다.

- 기준별 개수: `vib_rms` > 0.258 **1개**(normal_320, 길이 3), 세그먼트 AI0 rms > 정상 q99(0.1177) **6개**, 윈도우 1초 AI0 rms 최댓값 > 정상 윈도우 q99(0.1272) **21개**. 합집합 **26개**(정상 599의 4.3%), 세 기준을 모두 만족하는 세그먼트는 0개, 두 기준은 2개(normal_178: 세그먼트+윈도우, normal_320: vib_rms+세그먼트)다.
- 운전 상태: 26개 중 state0(고부하, 정상 전체의 42.1%)이 **25개(96.2%)**, state1은 1개(normal_321, 길이 3). 의심 세그먼트는 사실상 고부하 구간의 진동 상단 꼬리다 [데이터].
- 시간대(블록): 블록 0 6개 / 1 1개 / **2 8개 / 3 10개** / 4 1개(각 블록 정상 세그먼트 119~120개). 블록 2·3(00:35:01~00:59:28)에 18개(69%)가 몰린다.
- "확실 이상" 수준(`vib_rms` > 0.258)을 넘는 정상은 1개뿐이고 나머지 25개는 정상 상단 q99 부근이다. 이상 이벤트(확실 이상 18개)와 겹치는 정상은 사실상 1개 → 규칙 베이스라인의 세그먼트 오경보율 하한은 이 목록에서 나온다 [추정].
- **모델링 영향**: (1) 22·23에서 정상 상단 꼬리는 학습 정상에 포함하되 제거하지 않는다(라벨을 정상으로 신뢰). (2) 31에서 FP 사례 분석은 이 목록의 고부하·블록 2~3 세그먼트를 먼저 본다. (3) 시간 블록 분할에서 블록 3이 test일 때 이 10개가 오경보 원천이 될 수 있다.

## 3. E3 정상 하루 안 추이·드리프트

**가설**: 정상 77분 안에 부하 이동(블록 4의 state1 비율 상승)이 있고 피처 분포가 시간 블록마다 다르다. 상태를 층화해도 남는 추이가 있으면 "비정상(非定常) 피처"다. 시간축은 정렬용으로만 쓴다.

**방법**: (a) 1초 윈도우 정상을 `stats.segment_representatives`로 세그먼트 중앙값 1행으로 줄여 `stats.trend_table`(Spearman·Mann–Kendall)을 전체·state0·state1 각각 계산. (b) `stats.drift_ks_table`로 블록 k vs 이전 블록, 블록별 상태 비율 표. (c) 상태 전환 횟수·연속 지속 세그먼트 수·전환 직후 첫 세그먼트의 AI0 rms 효과(`stats.group_test_table`). 판정 기준: 층화 후 |ρ|>0.3 & p<0.01이면 비정상 피처.

In [4]:
CH = ["AI0_Vibration", "AI1_Vibration", "AI2_Current"]
E3COLS = [f"{c}_{m}" for c in CH for m in ("rms", "peak", "kurt", "crest")] + ["vib_rms_ratio", "cur_ac1", "cur_fit_r2"]
rep = stats.segment_representatives(win[win.label == 0], E3COLS)
rep = rep.merge(seg[["seg_uid", "start"]], on="seg_uid", how="left").sort_values("start").reset_index(drop=True)
print("정상 대표값 표:", rep.shape, "| n_windows 중앙값", rep["n_windows"].median(), "| state:", rep["state"].value_counts().to_dict(), "| block:", rep["block"].value_counts().sort_index().to_dict())
parts = []
for name, sub in [("all", rep), ("state0", rep[rep.state == 0]), ("state1", rep[rep.state == 1])]:
    t = stats.trend_table(sub, E3COLS, "start"); t.insert(0, "stratum", name); parts.append(t)
trend = pd.concat(parts, ignore_index=True)
trend["nonstationary_flag"] = (trend["spearman_rho"].abs() > 0.3) & (trend["spearman_p"] < 0.01)
trend.to_csv(RES / "normal_trend.csv", index=False)
w = trend.pivot(index="feature", columns="stratum", values=["spearman_rho", "spearman_p"]).reindex(E3COLS)
print(w.round(4).to_string())
strat = trend[trend.stratum != "all"].groupby("feature")["nonstationary_flag"].any()
print("층화(state0 또는 state1) 후 비정상 피처:", strat[strat].index.tolist())
print("전체에서만 플래그:", [f for f in trend[(trend.stratum == "all") & trend.nonstationary_flag]["feature"] if not strat[f]])

정상 대표값 표: (530, 25) | n_windows 중앙값 7.0 | state: {1.0: 304, 0.0: 226} | block: {np.int64(0): 113, np.int64(1): 102, np.int64(2): 99, np.int64(3): 108, np.int64(4): 108}


                    spearman_rho                 spearman_p                
stratum                      all  state0  state1        all  state0  state1
feature                                                                    
AI0_Vibration_rms        -0.2120  0.1825 -0.3174     0.0000  0.0059  0.0000
AI0_Vibration_peak       -0.2173  0.1843 -0.3315     0.0000  0.0054  0.0000
AI0_Vibration_kurt       -0.0324  0.1197 -0.0487     0.4574  0.0724  0.3973
AI0_Vibration_crest      -0.0824  0.0189 -0.0924     0.0579  0.7777  0.1078
AI1_Vibration_rms        -0.1910  0.2355 -0.2194     0.0000  0.0004  0.0001
AI1_Vibration_peak       -0.1903  0.2525 -0.2264     0.0000  0.0001  0.0001
AI1_Vibration_kurt        0.1103 -0.0137  0.0042     0.0110  0.8380  0.9414
AI1_Vibration_crest       0.1200  0.0274 -0.0088     0.0057  0.6821  0.8790
AI2_Current_rms          -0.1172  0.0491  0.0898     0.0069  0.4629  0.1182
AI2_Current_peak         -0.1236  0.0632  0.0649     0.0044  0.3439  0.2592
AI2_Current_

In [5]:
kparts = []
for name, sub in [("all", rep), ("state0", rep[rep.state == 0]), ("state1", rep[rep.state == 1])]:
    k_ = stats.drift_ks_table(sub, E3COLS, "block"); k_.insert(0, "stratum", name); kparts.append(k_)
ks = pd.concat(kparts, ignore_index=True); ks["significant_p<0.01"] = ks["ks_p"] < 0.01
ks.to_csv(RES / "block_drift_ks.csv", index=False)
print("KS 유의(p<0.01) 열 수 / 층·블록:"); print(ks.groupby(["stratum", "block"])["significant_p<0.01"].agg(["sum", "count"]).T.to_string())
print(ks[(ks.stratum == "all") & ks["significant_p<0.01"]].sort_values("ks_stat", ascending=False).head(10)[["feature", "block", "ks_stat", "ks_p", "levene_p"]].round(4).to_string(index=False))
sn = seg[seg.label == 0]; asg = split.block_assignment(sn, 5)
share = split.split_summary(sn, asg)
share.to_csv(RES / "block_state_share.csv"); print(share.to_string())

KS 유의(p<0.01) 열 수 / 층·블록:
stratum all             state0             state1            
block     1   2   3   4      1   2   3   4      1   2   3   4
sum       0   0   0  12      1   2   1   0      0   0   0   7
count    15  15  15  15     15  15  15  15     15  15  15  15
           feature  block  ks_stat  ks_p  levene_p
 AI1_Vibration_rms      4   0.5210   0.0    0.0000
AI0_Vibration_peak      4   0.5053   0.0    0.3302
        cur_fit_r2      4   0.5033   0.0    0.0000
 AI0_Vibration_rms      4   0.4980   0.0    0.5754
AI1_Vibration_peak      4   0.4820   0.0    0.0000
   AI2_Current_rms      4   0.4271   0.0    0.0000
  AI2_Current_peak      4   0.4197   0.0    0.0000
 AI2_Current_crest      4   0.3444   0.0    0.1722
     vib_rms_ratio      4   0.3441   0.0    0.0340
AI1_Vibration_kurt      4   0.2830   0.0    0.4762
       n_segments  n_outlier_seg  n_rows               start_min               start_max  share_state0.0  share_state1.0
block                                       

In [6]:
sq = seg[seg.label == 0].sort_values("start").reset_index(drop=True)
s = sq["state"].astype(int).to_numpy(); chg = np.r_[False, s[1:] != s[:-1]]
runs = pd.DataFrame({"state": s, "run": np.cumsum(chg)}).groupby("run").agg(state=("state", "first"), n_seg=("state", "size"))
sq["after_transition"] = chg.astype(int)
sq = sq.merge(rep[["seg_uid", "AI0_Vibration_rms", "AI2_Current_rms"]].rename(columns=lambda c: c if c == "seg_uid" else "rep_" + c), on="seg_uid")
tcols = ["rep_AI0_Vibration_rms", "rep_AI2_Current_rms"]
gt_parts = []
for name, sub in [("all", sq), ("state0", sq[sq.state == 0]), ("state1", sq[sq.state == 1])]:
    g = stats.group_test_table(sub, tcols, "after_transition"); g.insert(0, "stratum", name); gt_parts.append(g)
gt = pd.concat(gt_parts, ignore_index=True)
summ = [{"kind": "summary", "name": "n_segments_sequence", "value": len(s)}, {"kind": "summary", "name": "n_transitions", "value": int(chg.sum())},
        {"kind": "summary", "name": "n_segments_with_window_rep", "value": len(sq)}]
for st_ in (0, 1):
    r = runs[runs.state == st_]["n_seg"]
    summ += [{"kind": "run_length", "name": f"state{st_}_n_runs", "value": len(r)}, {"kind": "run_length", "name": f"state{st_}_median", "value": float(r.median())},
             {"kind": "run_length", "name": f"state{st_}_max", "value": int(r.max())}, {"kind": "run_length", "name": f"state{st_}_share_len1", "value": float((r == 1).mean())}]
out = pd.concat([pd.DataFrame(summ), gt.assign(kind="group_test", name=gt["stratum"] + ":" + gt["feature"])], ignore_index=True)
out.to_csv(RES / "state_transitions.csv", index=False)
print(pd.DataFrame(summ).to_string(index=False))
print("연속 지속 세그먼트 수 분포(run 길이 빈도):", runs["n_seg"].value_counts().sort_index().to_dict())
print(gt[["stratum", "feature", "n_a", "n_b", "median_a", "median_b", "mwu_p", "cliff_delta"]].round(4).to_string(index=False))

      kind                       name      value
   summary        n_segments_sequence 599.000000
   summary              n_transitions 325.000000
   summary n_segments_with_window_rep 530.000000
run_length              state0_n_runs 163.000000
run_length              state0_median   1.000000
run_length                 state0_max   5.000000
run_length          state0_share_len1   0.564417
run_length              state1_n_runs 163.000000
run_length              state1_median   1.000000
run_length                 state1_max  90.000000
run_length          state1_share_len1   0.595092
연속 지속 세그먼트 수 분포(run 길이 빈도): {1: 189, 2: 110, 3: 10, 4: 12, 5: 3, 7: 1, 90: 1}
stratum               feature  n_a  n_b  median_a  median_b  mwu_p  cliff_delta
    all rep_AI0_Vibration_rms  242  288    0.0598    0.0713 0.0000       0.2505
    all   rep_AI2_Current_rms  242  288   90.8282  111.2243 0.0003       0.1827
 state0 rep_AI0_Vibration_rms   81  145    0.0792    0.0788 0.9746      -0.0026
 state0   rep_

In [7]:
fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
cc = np.where(rep["state"] == 1, "tab:blue", "tab:orange")
for ax, col, lab in zip(axs, ["AI0_Vibration_rms", "AI2_Current_rms"], ["AI0 rms(세그먼트 중앙값)", "전류 AC rms(세그먼트 중앙값)"]):
    ax.scatter(rep["start"], rep[col], s=9, c=cc, alpha=.7); ax.set_ylabel(lab)
    for b in sorted(rep["block"].unique())[1:]:
        ax.axvline(rep.loc[rep.block == b, "start"].min(), c="k", ls=":", lw=1)
axs[0].set_title("정상 하루 타임라인 (파랑 state1 / 주황 state0, 점선 = 블록 경계)"); axs[1].set_xlabel("세그먼트 시작(정렬용 x축)")
paths.save_fig(FIG, "normal_timeline.png")

saved figures\04_data_profile_CHS\normal_timeline.png


**판정: 부분 채택** (`normal_trend.csv`, `block_drift_ks.csv`, `block_state_share.csv`, `state_transitions.csv`). 대표값은 1초 윈도우가 있는 정상 530 세그먼트(599 중 69개는 윈도우 없음).

**추이(층화 전후)** — 전체에서 |ρ|>0.3 & p<0.01인 피처는 없고(최대 |ρ| = `cur_fit_r2` −0.2410), 상태 층화 후 기준을 넘는 것은 **state1(저부하)의 `AI0_Vibration_rms` ρ −0.3174 (p<0.001)와 `AI0_Vibration_peak` ρ −0.3315 (p<0.001)** 두 개뿐이다. state0(고부하)에서는 같은 두 피처의 부호가 반대(+0.1825, +0.1843)이고 |ρ|<0.3이라 기준 미달이다. → **비정상(非定常) 피처 = state1 AI0 rms·peak**(약한 추이, 부호가 상태에 따라 뒤집힘) [데이터]. 다른 13개 피처는 층화 후에도 기준 미충족.

**블록 드리프트(KS, p<0.01 열 수 / 15열)** — 블록 1: 0, 블록 2: 0, 블록 3: 0, **블록 4: 12**. 블록 4 vs 이전 블록에서 `AI1_Vibration_rms` KS 0.5210, `cur_fit_r2` 0.5033, `AI0_Vibration_rms` 0.4980(중앙값 0.0709 → 0.0406), `AI2_Current_rms` 0.4271(113.0 → 87.8). 블록 4 정상은 **state1 비율 85.71%**(블록 0~3은 46.7~55.0%). 상태를 나눠도 state1에서 블록 4는 7열이 유의(KS 0.27~0.49)하고 state0은 블록 4 표본이 16개뿐이라 유의 0열(검정력 부족)이다 → 블록 4 드리프트는 **부하 상태 혼합 변화뿐 아니라 저부하 내부의 진동 감소도 포함** [데이터], 원인은 설비 상태 변화인지 운전 조건인지 이 데이터로 구분 불가 [추정].

**상태 전환** — 599 세그먼트 시퀀스에서 전환 **325회**, 상태별 연속 지속 세그먼트 수 중앙값 1(state0 최대 5, state1 최대 90; 길이 1 비율 56.4% / 59.5%). state1의 90개짜리 연속 구간은 블록 4의 저부하 구간이다. 전환 직후 첫 세그먼트의 AI0 rms는 state0으로 진입한 경우 차이 없음(중앙값 0.0792 vs 0.0788, p 0.975), state1로 진입한 경우 더 큼(0.0565 vs 0.0443, Cliff δ +0.274, p<0.001)이며, 전류 AC rms는 state0 진입 직후 더 큼(168.2 vs 144.5, δ +0.399, p<0.001). 다만 state1 비교군에 90개 연속 구간(진동 낮음)이 섞여 있어 전환 효과와 블록 4 드리프트가 교란된다(보류).

**모델링 영향**
- time_block 분할(22·23·24)에서 **블록 4 성능 하락은 표본 부족이 아니라 정상 분포 이동(부하·진동 하락)을 먼저 의심**한다. 블록 4의 오경보율 변화를 이상 탐지 실패로 해석하지 않는다.
- 상태별 임계·정규화(상태 조건부)가 필요한 근거가 하나 더 생긴다: state1 AI0 rms는 시간 따라 감소.
- 현장 활용(4번): 저부하 장시간 진입 후에는 임계 재적합 또는 상태별 기준선 갱신이 필요하다는 근거(재적합 주기 제안은 확정 전 [추정]). 전환이 잦아(325회) 전환 직후 구간을 별도 처리할 실익은 근거 부족(보류).

## 4. E4 이상 이벤트 추이 재검 (AC 기준)

**가설**: 02 §4의 "전류 악화 추세"(전류 DC ρ +0.66, p 0.001)는 DC 오프셋이라 파일 형식 누수와 얽혀 있다. 평균 제거 후 AC 피처(진동 rms·전류 AC rms·`vib_corr01`·`cur_fit_r2`)로 보면 추세 결론이 바뀐다.

**방법**: 이상 21 세그먼트를 시작 시각 순으로 놓고, `segments.csv`의 원시 `vib_rms`(02 §4 재현)와 1초 윈도우 세그먼트 대표값(윈도우가 있는 세그먼트만; `n` 열)에 `stats.trend_table`. 누수 채널 `AI2_Current_dc`는 참고 행으로 나란히 둔다. 판정 기준: AC 기준 유의 추세가 없으면 "점진적 열화" 서술을 삭제하고 "단일 이벤트, 진행 방향 판정 불가"로 확정.

In [8]:
E4W = ["AI0_Vibration_rms", "AI1_Vibration_rms", "AI2_Current_rms", "vib_corr01", "cur_fit_r2", "vib_rms_ratio"]
osg = seg[seg.label == 1].sort_values("start").reset_index(drop=True)
orep = stats.segment_representatives(win[win.label == 1], E4W).merge(osg[["seg_uid", "seg_id", "start"]], on="seg_uid")
orep = orep.sort_values("start").reset_index(drop=True)
print("이상 세그먼트:", len(osg), "| 윈도우 있는 세그먼트:", len(orep), "| 윈도우 없는 세그먼트 seg_id:", sorted(set(osg.seg_id) - set(orep.seg_id)))
t_raw = stats.trend_table(osg, ["vib_rms", "AI2_Current_rms", "AI2_Current_dc"], "start")
t_raw["source"] = "segments.csv(원시 세그먼트 피처, 02 §4 재현)"
t_raw.loc[t_raw.feature == "AI2_Current_dc", "feature"] = "AI2_Current_dc [참고: 누수 채널]"
t_raw.loc[t_raw.feature == "AI2_Current_rms", "feature"] = "AI2_Current_rms(원시, DC 포함)"
t_win = stats.trend_table(orep, E4W, "start"); t_win["source"] = "parquet 1초 윈도우 세그먼트 중앙값(표준 전처리 후)"
t_win.loc[t_win.feature == "AI2_Current_rms", "feature"] = "AI2_Current_rms(AC, 평균 제거)"
ot = pd.concat([t_raw, t_win], ignore_index=True)
ot["significant_p<0.05"] = ot["spearman_p"] < 0.05
ot.to_csv(RES / "outlier_trend_ac.csv", index=False)
print(ot[["source", "feature", "n", "spearman_rho", "spearman_p", "mk_p", "slope_per_unit"]].round(4).to_string(index=False))

이상 세그먼트: 21 | 윈도우 있는 세그먼트: 17 | 윈도우 없는 세그먼트 seg_id: [0, 5, 9, 10]
                            source                    feature  n  spearman_rho  spearman_p   mk_p  slope_per_unit
segments.csv(원시 세그먼트 피처, 02 §4 재현)                    vib_rms 21       -0.4299      0.0518 0.0655         -2.6815
segments.csv(원시 세그먼트 피처, 02 §4 재현) AI2_Current_rms(원시, DC 포함) 21        0.5779      0.0061 0.0122        800.9596
segments.csv(원시 세그먼트 피처, 02 §4 재현) AI2_Current_dc [참고: 누수 채널] 21        0.6597      0.0011 0.0034       2031.3539
 parquet 1초 윈도우 세그먼트 중앙값(표준 전처리 후)          AI0_Vibration_rms 17       -0.3382      0.1842 0.1740         -1.5777
 parquet 1초 윈도우 세그먼트 중앙값(표준 전처리 후)          AI1_Vibration_rms 17       -0.4436      0.0745 0.0638         -0.9706
 parquet 1초 윈도우 세그먼트 중앙값(표준 전처리 후) AI2_Current_rms(AC, 평균 제거) 17       -0.3333      0.1911 0.1082       -188.9001
 parquet 1초 윈도우 세그먼트 중앙값(표준 전처리 후)                 vib_corr01 17        0.3431      0.1775 0.2016          2.8228
 parquet 1초 윈도우 세그먼트 중

In [9]:
fig, axs = plt.subplots(2, 2, figsize=(11, 6.5))
ORD = {u: i for i, u in enumerate(osg["seg_uid"])}; orep["k"] = orep["seg_uid"].map(ORD); osg["k"] = np.arange(len(osg))
quiet = {3, 19, 20}
for ax, (df_, col, lab) in zip(axs.ravel(), [(orep, "AI0_Vibration_rms", "AI0 rms"), (orep, "AI2_Current_rms", "전류 AC rms"), (orep, "vib_corr01", "vib_corr01"), (osg, "AI2_Current_dc", "[참고: 누수 채널] 전류 DC")]):
    m = df_["seg_id"].isin(quiet)
    ax.plot(df_["k"], df_[col], "o-", c="tab:blue", ms=4, lw=.8); ax.scatter(df_.loc[m, "k"], df_.loc[m, col], c="r", s=45, zorder=3, label="3·19·20")
    ax.set_ylabel(lab); ax.set_xlabel("이상 세그먼트 순서(시작 시각순)"); ax.legend(fontsize=8)
fig.suptitle("이상 이벤트 시간 진행 (AC 기준 3패널 + DC 참고)")
paths.save_fig(FIG, "outlier_timeline_ac.png")

saved figures\04_data_profile_CHS\outlier_timeline_ac.png


**판정: 채택** (`outlier_trend_ac.csv`) — AC 기준 유의 추세가 없으므로 02 §4의 "점진적 열화" 서술을 삭제하고 **"단일 이벤트, 진행 방향 판정 불가"** 로 확정한다. 윈도우 대표값이 있는 이상 세그먼트는 17개(21개 중, seg_id 0·5·9·10 제외)이며 표에 `n`으로 표기했다.

| 피처 | n | Spearman ρ | p | 비고 |
|---|---|---|---|---|
| `vib_rms`(원시, 02 §4 재현) | 21 | −0.4299 | 0.0518 | 02 §4와 동일(−0.430 / 0.052) |
| 전류 RMS(원시, DC 포함) | 21 | +0.5779 | 0.0061 | DC 성분이 지배 |
| **[참고: 누수 채널] 전류 DC** | 21 | **+0.6597** | **0.0011** | 02 §4 재현(+0.66 / 0.001) |
| AI0 rms(AC) | 17 | −0.3382 | 0.1842 | 유의 아님 |
| AI1 rms(AC) | 17 | −0.4436 | 0.0745 | 유의 아님(경계) |
| 전류 AC rms | 17 | −0.3333 | 0.1911 | **부호가 DC 기준과 반대** |
| `vib_corr01` | 17 | +0.3431 | 0.1775 | 유의 아님 |
| `cur_fit_r2` | 17 | +0.2770 | 0.2818 | 유의 아님 |
| `vib_rms_ratio` | 17 | −0.0098 | 0.9702 | 유의 아님 |

- 평균 제거(AC) 기준 6개 피처 모두 p>0.05, 가장 작은 p는 AI1 rms 0.0745. 02 §4의 전류 "악화" 추세(DC ρ +0.6597, 원시 RMS ρ +0.5779)는 DC 오프셋에서 나온 것이고 AC 전류는 반대 부호(−0.3333, p 0.1911)다 → 파일 형식 누수·오프셋과 얽힌 DC 추세는 설비 열화 근거로 쓸 수 없다 [데이터].
- 한계: n=17로 검정력이 낮아 "추세 없음"이 아니라 **"추세를 확인할 수 없음"** 이다. 이벤트 1건이고 첫 세그먼트가 길이 4로 잘려 시작점도 불명 [추정].
- **모델링 영향**: 시간 추세 파생 피처("누적 악화 점수") 제외. 조기탐지 스토리는 "점진적 열화 예지"가 아니라 "이상 구간 진입 후 몇 초 안에 탐지"(탐지 지연 지표)로 잡는다. 3·19·20은 표준 전처리 후 진동·전류가 조용하다는 21 V2-a 결과와 같은 방향이므로 31에서 FN을 라벨 노이즈 후보로 분리해 본다.

## 5. 결론

**판정 요약**

| 절 | 판정 | 핵심 수치 |
|---|---|---|
| B-1 타임스탬프 | 채택 | 단조 증가, 실효 fs 10.0000 Hz(편차 0.0%), 0.1 s ±5% 밖 0.0052%(정상)·0%(이상), 중복 1행은 같은 값 |
| B-2 정상 의심 | 채택 | 26개(4.3%): 기준별 vib_rms 1 / 세그먼트 q99 6 / 윈도우 q99 21, state0 25개, 블록 2·3에 18개 |
| E3 추이·드리프트 | 부분 채택 | 층화 후 비정상 피처 2개(state1 AI0 rms ρ −0.3174, peak ρ −0.3315), 블록 4 KS 유의 12/15열(state1 층화 7열), 전환 325회 |
| E4 이상 AC 추세 | 채택 | AC 6피처 모두 p>0.05(최소 0.0745), DC 참고 ρ +0.6597 p 0.0011 |

**확정 페이지로 승격할 문장 후보** (실험 페이지 판정 "채택", 결과가 저장소 CSV로 재현될 때만)

- 타임스탬프는 정상·이상 파일 모두 단조 증가하고 burst 안 실효 샘플링 주파수는 10.0000 Hz(편차 0.0%)라, 에일리어싱 계산(60 Hz → 0.6 Hz)의 fs 전제가 성립한다. (`timestamp_integrity.csv`, [데이터])
- 정상 599 세그먼트 중 26개(4.3%)는 진동 상단 꼬리 의심 세그먼트이며 25개가 고부하(state0), 18개가 시각 00:35~00:59 구간(블록 2·3)에 몰려 있다 — FP 분석 후보. (`normal_suspects.csv`, [데이터])
- 정상 하루 안 시간 추이는 상태 층화 후 저부하(state1)의 AI0 rms·peak(ρ −0.3174 / −0.3315)에서만 나타나고, 마지막 시간 블록(블록 4)은 이전 블록 대비 15개 피처 중 12개의 분포가 달라진다(KS p<0.01). 시간 블록 분할의 마지막 블록 성능 변화는 정상 분포 이동을 먼저 의심한다. (`normal_trend.csv`, `block_drift_ks.csv`, [데이터])
- 02 §4의 전류 "악화 추세"는 DC 기준(ρ +0.6597, p 0.0011)이며 평균 제거 후 AC 기준으로는 진동·전류 어느 피처도 유의 추세가 없다(모두 p>0.05, n=17). 이상 이벤트는 단일 이벤트이며 진행 방향은 판정할 수 없다. (`outlier_trend_ac.csv`, [데이터])
- 보류: 블록 4 드리프트의 원인(설비 상태 vs 운전 조건)과 전환 직후 효과는 이 데이터로 구분 불가 [추정].